In [1]:
import torch
import wandb as wb
import yaml
import itertools
import time

from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from model import CMCDNet
from loss import LossFunction
from dataset import ChangeDetctionDataset
from transform import PairedTransform, ValTransform
from sampler import BucketBatchSampler
from utils.logging import log_images
from utils.eval_metrics import iou_score, precision_recall, f1_score, find_best_threshold

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

In [2]:
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

data = cfg["data"]
hyperparameter = cfg["training"]
aug = cfg["augmentations"]

In [3]:
### Train Dataset
dataset = ChangeDetctionDataset(
    pre_img_path=data["train"]["pre_event"],
    post_img_path=data["train"]["post_event"],
    target_img_path=data["train"]["target"],
    patch_size=data["patch_size"],
    stride=data["stride"],
    index_path=data["index_path"],
    build_metadata=True,
    transform=PairedTransform(
        horizontal_flip_p=aug["horizontal_flip"]["probability"],
        vertical_flip_p=aug["vertical_flip"]["probability"]
    )
)

batch_sampler = BucketBatchSampler(
    dataset.patch_metadata,
    batch_size = hyperparameter["batch_size"],
    ratio=cfg["sampler"]["ratios"],
)

train_loader = DataLoader(
    dataset,
    batch_sampler=batch_sampler,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True)

In [4]:
### Validaation Dataset
validation_dataset = ChangeDetctionDataset(
    pre_img_path=data["val"]["pre_event"],
    post_img_path=data["val"]["post_event"],
    target_img_path=data["val"]["target"],
    patch_size=data["patch_size"],
    stride=data["stride"],
    index_path=None,
    build_metadata=False,
    transform=ValTransform()
)

val_loader = DataLoader(
    validation_dataset,
    batch_size=hyperparameter["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")    
    scaler = GradScaler("cuda")

model = CMCDNet().to(device)
criterion = LossFunction()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=hyperparameter["learning_rate"],
    weight_decay=hyperparameter["weight_decay"], 
)

size = sum(1 for m in dataset.patch_metadata if m["bucket"] != "discard")

steps_per_epoch = size // hyperparameter["batch_size"] ##15827.875
global_step = 0
# loss_tracker = {}

Using GPU: NVIDIA RTX A5000


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [6]:
import os

with wb.init(project="EO-SAR Change-Detection", name="cmcdnet-run-001", config=cfg) as run:
    best_val_iou = float("-inf")
    ckpt_dir = "checkpoints"

    for epoch in range(hyperparameter["epochs"]):
        start = time.perf_counter()
        model.train()
        running = 0.0
        pbar = tqdm(itertools.islice(train_loader, steps_per_epoch), total=steps_per_epoch, leave=False)
        for step_idx, (pre, post, target, _) in enumerate(pbar):
            pre = pre.to(device)
            post = post.to(device)
            target = target.to(device)

            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(pre, post)
                loss = criterion(logits, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running += loss.item()
            global_step += 1

            current_step = step_idx + 1
            current_lr = optimizer.param_groups[0]['lr']
            

            pbar.set_description(f"Epoch {epoch+1}")
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{current_lr:.2e}"})

            if global_step % cfg["metrics"]["log_steps"] == 0:
                iou = iou_score(logits, target)
                precision, recall = precision_recall(logits, target)
                f1 = f1_score(precision, recall)

                log = {
                    "train/loss": loss.item(),
                    "train/iou": iou,
                    "train/precision": precision,
                    "train/recall": recall,
                    "train/f1_score": f1,
                    "train/learning_rate": current_lr
                }

                run.log(log, step=global_step)

            if cfg["metrics"]["log_images"] and global_step % cfg["metrics"]["image_log_every"] == 0:
                pred = (torch.sigmoid(logits) > 0.5).squeeze(1)
                log_images(pre, post, pred, target, step=global_step, max_images=2)

        print()
        model.eval()
        with torch.no_grad():
            val_loss = 0.0
            iou_loss = 0.0
            total_tp, total_fp, total_fn = 0, 0, 0
            for pre, post, target, _ in val_loader:
                pre = pre.to(device)
                post = post.to(device)
                target = target.to(device)
                logits = model(pre, post)
                loss = criterion(logits, target)
                val_loss += loss.item()
                iou_loss += iou_score(logits, target)

                pred = (torch.sigmoid(logits) > 0.5)
                target_mask = target.unsqueeze(1) if target.dim() == 3 else target
                target_mask = target_mask.bool()
                tp = (pred & target_mask).sum().item()
                fp = (pred & ~target_mask).sum().item()
                fn = (~pred & target_mask).sum().item()
                total_tp += tp
                total_fp += fp
                total_fn += fn

            precision = total_tp / (total_tp + total_fp + 1e-3)
            recall = total_tp / (total_tp + total_fn + 1e-3)
            f1 = (2 * precision * recall) / (precision + recall + 1e-3)
            val_avg_loss = val_loss / len(val_loader)
            val_avg_iou = iou_loss / len(val_loader)
            train_epoch_loss = running / steps_per_epoch

            val_log = {
                "val/loss": val_avg_loss,
                "val/iou": val_avg_iou,
                "val/precision": precision,
                "val/recall": recall,
                "val/f1_score": f1,
                "val/learning_rate": current_lr
            }

            run.log(val_log, step=epoch)
            if val_avg_iou > best_val_iou:
                best_val_iou = val_avg_iou
                ckpt_path = os.path.join(ckpt_dir, f"best_epoch{epoch+1}_iou{best_val_iou:.4f}.pth")
                torch.save({
                    "epoch": epoch,
                    "global_step": global_step,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict() if 'scaler' in globals() else None,
                    "best_val_iou": best_val_iou,
                }, ckpt_path)
                print(f"Saved new best checkpoint: {ckpt_path}")

                artifact = wb.Artifact(
                    name="cmcdnet-best",
                    type="model",
                    metadata={"epoch": epoch + 1, "best_val_iou": float(best_val_iou)}
                )
                artifact.add_file(ckpt_path)
                run.log_artifact(artifact)
                artifact.wait() 

        epoch_time = time.perf_counter() - start
        print(
            f"epoch {epoch + 1}/{hyperparameter['epochs']} | "
            f"train_loss {train_epoch_loss:.4f} | "
            f"val_loss {val_avg_loss:.4f} | "
            f"val_iou {val_avg_iou:.4f} | "
            f"val_f1 {f1:.4f} | "
            f"time {epoch_time:.2f}s"
        )
        run.log({"train/per_epoch_sec": epoch_time}, step=epoch)
        run.log({"train/epoch_loss": train_epoch_loss}, step=steps_per_epoch)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: karanjayakumar (karanjayakumar-freelancer) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  0%|          | 0/7913 [00:00<?, ?it/s]


Saved new best checkpoint: checkpoints/best_epoch1_iou0.0493.pth


wandb: WARNING Tried to log to step 0 that is less than the current step 7500. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


epoch 1/10 | train_loss 0.6505 | val_loss 0.3414 | val_iou 0.0493 | val_f1 0.5787 | time 4275.78s


...


epoch 9/10 | train_loss 0.2727 | val_loss 0.2737 | val_iou 0.0524 | val_f1 0.6199 | time 6969.96s


  0%|          | 0/7913 [00:00<?, ?it/s]

wandb: WARNING Tried to log to step 8 that is less than the current step 71000. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 7913 that is less than the current step 71000. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.



epoch 10/10 | train_loss 0.2706 | val_loss 0.2708 | val_iou 0.0479 | val_f1 0.6092 | time 4785.02s


train/epoch_loss,▁
train/f1_score,▁▂▃▅▄▆▅▅▆▆▆▆▄▅▆▅▅▇▆▆▆▇█▅██▆▆▆▁▅▆▄▆▆▅▇▆▇▇
train/iou,▃▂▂▃▃▁▆▄▆▃▆▅▆▇▇▄▅▆▆▇▅▅█▄▆▇██▆▂▇▇▇▇▆▇▆█▆▇
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,▅▇▅█▃▂▂▄▂▄▃▃▁▅▃▂▁▃▁▃▂▃▁▄▃▂▃▂▂▃▄▅▂▃▂▂▂▃▂▂
train/precision,▄▁▅▄▄▃▄▃▆▃▆▆▅▅▆▄▆▅▅▆▆▆▆▃█▇▅▇▆▆▆▃▆▇▆▅▆▆▄▅
train/recall,▁▃▃▁▃▃▄▄▆▆▅▃▅▃▄▅▅▅▆▆▅▇▃▇█▅▆▄▅█▆▆▅▄▆▆▆▇▆▇
train/epoch_loss,0.65049
train/f1_score,0.46673
train/iou,0.39335
train/learning_rate,0.0001


In [7]:
print(torch.cuda.is_available())

True


In [8]:
# Example: Save model and optimizer states
torch.save({
'epoch': 10,
'model_state_dict': model.state_dict(),
'optimizer_state_dict': optimizer.state_dict(),
'loss': loss.item(),
}, "model_checkpoint.pth")

In [6]:
# Threshold sweep EDA (run after training or with a loaded checkpoint)
torch.load("model_checkpoint.pth", map_location=torch.device('cpu'))

model.eval()
val_logits = []
val_targets = []
with torch.no_grad():
    for pre, post, target, _ in val_loader:
        pre = pre.to(device)
        post = post.to(device)
        target = target.to(device)
        logits = model(pre, post)
        val_logits.append(logits.detach().cpu())
        val_targets.append(target.detach().cpu())

val_logits = torch.cat(val_logits, dim=0)
val_targets = torch.cat(val_targets, dim=0)

best_thr_f1, best_f1 = find_best_threshold(val_logits, val_targets, metric="f1")
best_thr_iou, best_iou = find_best_threshold(val_logits, val_targets, metric="iou")

print(f"Best threshold by F1: {best_thr_f1:.2f} (F1={best_f1:.4f})")
print(f"Best threshold by IoU: {best_thr_iou:.2f} (IoU={best_iou:.4f})")

thresholds = torch.linspace(0.05, 0.95, steps=19)
rows = []
for t in thresholds:
    thr = float(t)
    precision, recall = precision_recall(val_logits, val_targets, threshold=thr)
    f1 = f1_score(precision, recall)
    iou = iou_score(val_logits, val_targets, threshold=thr)
    rows.append((thr, f1, iou, precision, recall))

rows = sorted(rows, key=lambda x: x[1], reverse=True)
print("Top thresholds by F1:")
for thr, f1, iou, precision, recall in rows[:5]:
    print(f"thr={thr:.2f} f1={f1:.4f} iou={iou:.4f} p={precision:.4f} r={recall:.4f}")

Best threshold by F1: 0.45 (F1=0.0332)
Best threshold by IoU: 0.45 (IoU=0.0227)
Top thresholds by F1:
thr=0.50 f1=0.0389 iou=0.0220 p=0.0238 r=0.1122
thr=0.45 f1=0.0387 iou=0.0227 p=0.0227 r=0.1359
thr=0.05 f1=0.0387 iou=0.0227 p=0.0227 r=0.1360
thr=0.10 f1=0.0387 iou=0.0227 p=0.0227 r=0.1360
thr=0.15 f1=0.0387 iou=0.0227 p=0.0227 r=0.1360
